# 1. Khai báo thư viện

In [ ]:
!pip install googlesearch-python

In [ ]:
import numpy as np
import pandas as pd
import ipaddress
import os.path
import requests
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from urllib.parse import urlparse,urlencode
from tld import get_tld
from googlesearch import search
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, f1_score, recall_score, precision_score, roc_curve, auc

## 2. Chuẩn bị dữ liệu

In [ ]:
file0 = '../Data/legit_url.csv'
file1 = '../Data/verified_online.csv'


## 2.1. Legit URLs

In [ ]:
data0 = pd.read_csv(file0)
data0.head()

In [ ]:
data0.shape

In [ ]:
# Lấy ngẫu nhiên 15000 mẫu
legiturl = data0.sample(n=15000, random_state = 12).copy()
legiturl = legiturl.reset_index(drop=True)
legiturl.head()

In [ ]:
legiturl.shape
legiturl['Label'] = 0
legiturl.head()

## 2.2. Phishing URLs

In [ ]:
data1 = pd.read_csv(file1)
data1.head()

In [ ]:
data1.shape

In [ ]:
#Lấy ngẫu nhiên 5000 mẫu
phishurl = data1 .sample(n = 15000, random_state = 12).copy()
phishurl = phishurl.reset_index(drop=True)
phishurl.head()

In [ ]:
phishurl = phishurl[['url']]
phishurl.shape

In [ ]:
phishurl['Label'] = 1
phishurl.head()

## 2.3. Gộp dữ liệu

In [ ]:
urldata = pd.concat([legiturl, phishurl]).reset_index(drop=True)
urldata.shape

In [ ]:
urldata.head()

In [ ]:
urldata.Label.value_counts()

In [ ]:
urldata.info()

In [ ]:
df = urldata
df.head()

In [ ]:
df.to_csv("urldata.csv", index=False, encoding="utf-8")

In [ ]:
# Đếm số dữ liệu trùng lặp
print ('Số dòng bị trùng lặp: ',df.duplicated().sum())

# Xóa dữ liệu trùng lặp
df = df.drop_duplicates()
print ('Số dòng trùng lặp sau khi xử lý: ',df.duplicated().sum())

# 3. Trích xuất đặc trưng

## 3.1. Domain

In [ ]:
# 1. Lấy Domain của URL
def domain (url):
  domain = urlparse(url).netloc
  if re.match(r"^www.", domain):
    domain = domain.replace("www.","")
  return domain

df['Domain'] = df['url'].apply(lambda i: domain(i))
df.head()

## 3.2. Kiểm tra địa chỉ IP trong URL

* Nếu URL sử dụng địa chỉ IP thay vì tên miền (ví dụ: http://125.98.3.123) thì có thể đó là phishing.
* Nếu URL có tên miền hợp lệ bình thường (ví dụ: http://www.example.com) thì đó là legit

In [ ]:
# 2. Kiểm tra IP
def has_ip_address(url):
  try:
    ipaddress.ip_address(urlparse(url).hostname)
    return 1 # Phishing
  except (TypeError, ValueError):
    return 0 # Legitimate
  
df['Having_IP'] = df['url'].apply(has_ip_address)
df.head()

## 3.3. Độ dài URL

Phishing URL thường dài hơn để che giấu những phần đáng ngờ thanh địa chỉ

In [ ]:
# 3. Độ dài URL
def URLlength(url):
    # Đo độ dài của URL
    url_length = len(url)

    # Trả về độ dài URL mà không phân loại
    return url_length

df['URL_Length'] = df['url'].apply(lambda i: URLlength(i))
df.head()

## 3.4. Rút gọn URL

Một vài URL sẽ sử dụng phương pháp rút gọn tạo ra những URL mới ngắn hơn để dễ dàng chia sẻ. Tuy nhiên, đây cũng là cách các PhisURL "núp bóng". Do đó:
* Nếu URL rút gọn (ví dụ: bit.ly, goo.gl, tinyurl, v.v.) -> PhishURL
* Nếu URL không rút gọn -> LegitURL

In [ ]:
shortening_services = r"bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|" \
                      r"yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|" \
                      r"short\.to|BudURL\.com|ping\.fm|post\.ly|Just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|" \
                      r"doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|t\.co|lnkd\.in|db\.tt|" \
                      r"qr\.ae|adf\.ly|goo\.gl|bitly\.com|cur\.lv|tinyurl\.com|ow\.ly|bit\.ly|ity\.im|q\.gs|is\.gd|" \
                      r"po\.st|bc\.vc|twitthis\.com|u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|x\.co|" \
                      r"prettylinkpro\.com|scrnch\.me|filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|" \
                      r"tr\.im|link\.zip\.net"

In [ ]:
# 4. URL rút gọn
def tinyURL(url):
    match = re.search(shortening_services, url)  # Tìm kiếm nếu URL có dịch vụ rút gọn
    if match:
        return 1  # Phishing
    else:
        return 0  # Legitimate
    
df ['Tiny_URL'] = df['url'].apply(lambda i :tinyURL(i))
df.head()

## 3.5. Độ dài của TLD

TLD (Top-Level Domain) là phần cuối cùng của một tên miền trong hệ thống phân cấp của DNS (Domain Name System). Ví dụ, trong URL www.example.com, phần .com là TLD. Do đó:
* Nếu độ dài của TLD lớn hơn 3 ký tự -> PhishURL
* Nếu độ dài của TLD nhở hơn hoặc bằng 3 ký tự -> LegitURL

In [ ]:
# 5. Độ dài TLD
def tld_length(url):
  try:
      # Lấy TLD của URL
      tld = get_tld(url, fail_silently=True)

      # Nếu TLD tồn tại, trả về độ dài của TLD
      if tld:
          return len(tld)  # Đếm độ dài TLD

      else:
          return 0  # Nếu không có TLD, trả về 0

  except (TypeError, ValueError):
      return 0  # Phản hồi 0 nếu có lỗi trong việc lấy TLD
  
df['TLD_Length'] = df['url'].apply(tld_length)
df.head()

## 3.6. Số ký tự chữ số trong URL




URL có chứa nhiều ký tự số thì có có thể là PhishURL

In [ ]:
# 6. Đếm số lượng ký tự số có trong URL
def digitCount(url):
    # Đếm số lượng chữ số trong URL
    count = 0
    for char in url:
        if char.isnumeric():
            count += 1
    return count

df['Digit_Count'] = df['url'].apply(lambda i : digitCount(i))
df.head()

## 3.7. Số chữ cái trong URL

PhishURL thường có nhiều chữ cái hơn để che giấu tên miền đáng ngờ

In [ ]:
# 7. Đếm số lượng chữ cái có trong URL
def letterCount(url):
    letters = 0
    for i in url:
        if i.isalpha():
            letters = letters + 1
    return letters

df['Letter_Count'] = df['url'].apply(lambda i : letterCount(i))
df.head()

## 3.8. Số lượng dấu '.'

In [ ]:
# 8. Đếm số dấu '.'
def dotCount(url):
    return url.count('.')

df['Dot_Count'] = df['url'].apply(lambda i : dotCount(i))
df.head()

## 3.9. Số lượng 'www'

In [ ]:
# 9. Đếm 'www'
def wwwCount(url):
    return url.count('www')

df['www_Count'] = df['url'].apply(lambda i : wwwCount(i))
df.head()

## 3.10. Số lượng '@'

In [ ]:
#10. Đếm số @ trong URL
def atSignCount(url):
    return url.count('@')

df['At_Count'] = df['url'].apply(lambda i : atSignCount(i))
df.head()

## 3.11. Số lượng '-'

In [ ]:
# 11. Đếm '-' trong domain URL
def hyphenCount(url):
    return url.count('-')

df['Hyphen_Count'] = df['url'].apply(lambda i : hyphenCount(i))
df.head()

## 3.12. Số lượng dấu %

In [ ]:
# 12. Đếm '%'
def perCount(url):
    return url.count('%')

df['Per_Count'] = df['url'].apply(lambda i : perCount(i))
df.head()

## 3.13. Số lượng dấu '?'

In [ ]:
# 13. Đếm '?'
def quesCount(url):
    return url.count('?')

df['Ques_Count'] = df['url'].apply(lambda i : quesCount(i))
df.head()

## 3.14. Số lượng dấu '='

In [ ]:
# 14. Đếm dâu '='
def equalCount(url):
    return url.count('=')

df['Equal_Count'] = df['url'].apply(lambda i : equalCount(i))
df.head()

## 3.15. Vị trí của dấu '//'

In [ ]:
# 15. Số lượng dấu '//'
def redirection(url):
    pos = url.rfind('//')

    # Kiểm tra vị trí của '//' trong URL
    if pos > 6: 
        if pos > 7:
            return 1  
        else:
            return 0  
    else:
        return 0 

df['Redirection'] = df['url'].apply(lambda i : redirection(i))
df.head()

## 3.16. Độ sâu của URL

Các URL hợp lệ thường có ít thư mục con, trong khi đó các URL lừa đảo sẽ có nhiều hơn

In [ ]:
def getDepth(url):
    s = urlparse(url).path.split('/')
    depth = 0
    for j in range(len(s)):
        if len(s[j]) != 0:
            depth = depth+1
    return depth

df['Depth'] = df['url'].apply(lambda i : getDepth(i))
df.head()

## 3.18. Kiểm tra "http/https" trong Domain

Các web lừa đảo thường sử dụng 'https' hoặc 'http' trong domain để đánh lừa người dùng. Do đó:
* Nếu domain URL có chứa 'http' hoặc 'https' -> PhishURL
* Nếu domain URL không có -> LegitURL

In [ ]:
# 18. Kiểm tra "http/https"
def httphttpsDomain(url):
    domain = urlparse(url).netloc

    if 'http' in domain or 'https' in domain:
        return 1  # Phishing
    else:
        return 0  # Legit

df['httphttpsDomain'] = df['url'].apply(lambda i : httphttpsDomain(i))
df.head()


## 3.19. Độ dài tên của thư mục đầu tiên

In [ ]:
# 19. Độ dài của thư mục đầu tiên
def first_directory_length(url):
    parts = [part for part in urlparse(url).path.split('/') if part]
    return len(parts[0]) if parts else 0
    
df['FD_Length'] = df['url'].apply(first_directory_length)
df.head()

# 4. Trực quan hóa dữ liệu

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df. info()

## 4.1. Đơn biến

### 4.1.1. Binary cols

In [ ]:
# Having_IP
sns.countplot(x='Having_IP', data = df, palette='Set2')
plt.show()

In [ ]:
# Tiny_URL 
sns.countplot(x='Tiny_URL', data = df, palette='Set2')
plt.show()

In [ ]:
# Google Index
sns.countplot(x='httphttpsDomain', data=df, palette='Set2')
plt.show()

In [ ]:
# redirection
sns.countplot(x='Redirection', data=df, palette='Set2')
plt.show()

In [ ]:
# Label
sns.countplot(x='Label', data=df, palette='Set2')
plt.show()

### 4.1.2. Univariate cols

In [ ]:
# URL_Length
sns.histplot(df['URL_Length'], bins = 30)
plt.title('URL_Length')
plt.grid(True)
plt.show()

In [ ]:
# TLD_Length
sns.histplot(df['TLD_Length'], bins = 30)
plt.title('TLD_Length')
plt.grid(True)
plt.show()

In [ ]:
# Digit_Count
sns.histplot(df['Digit_Count'], bins = 30)
plt.title('Digit_Count')
plt.grid(True)
plt.show()

In [ ]:
# Letter_Count
sns.histplot(df['Letter_Count'], bins = 30)
plt.title('Letter_Count')
plt.grid(True)
plt.show()

In [ ]:
# Dot_Count
sns.histplot(df['Dot_Count'], bins = 30)
plt.title('Dot_Count')
plt.grid(True)
plt.show()

In [ ]:
# www_Count
sns.histplot(df['www_Count'], bins = 30)
plt.title('www_Count')
plt.grid(True)
plt.show()

In [ ]:
# At_Count
sns.histplot(df['At_Count'], bins = 30)
plt.title('At_Count')
plt.grid(True)
plt.show()

In [ ]:
# Hyphen_Count
sns.histplot(df['Hyphen_Count'], bins = 30)
plt.title('Hyphen_Count')
plt.grid(True)
plt.show()

In [ ]:
# Per_Count
sns.histplot(df['Per_Count'], bins = 30)
plt.grid(True)
plt.title('Per_Count')
plt.grid(True)
plt.show()

In [ ]:
# Ques_Count
sns.histplot(df['Ques_Count'], bins = 30)
plt.title('Ques_Count')
plt.grid(True)
plt.show

In [ ]:
# Equal_Count
sns.histplot(df['Equal_Count'], bins = 30)
plt.title('Equal_Count')
plt.grid(True)
plt.show()

In [ ]:
# Depth
sns.histplot(df['Depth'], bins = 30)
plt.title('Depth')
plt.grid(True)
plt.show()

In [ ]:
# FD_Length
sns.histplot(df['FD_Length'], bins = 30)
plt.title('FD_Length')
plt.grid(True)
plt.show()

Một vài đặc trưng chỉ có một giá trị, không có ý nghĩa để xây dựng mô hình dự đoán => drop

In [ ]:
df = df.drop(['url', 'Domain', 'httphttpsDomain'], axis = 1)
df.head()

## 4.2. Đa biến

In [ ]:
plt.figure(figsize=(15,13))

sns.heatmap(
    df.corr(),                 # ma trận tương quan
    annot=True,                # hiển thị số
    fmt=".2f",                 # 2 chữ số thập phân
    cmap="coolwarm",           # bảng màu (có thể thử 'YlGnBu', 'viridis', 'magma', v.v.)
    linewidths=0.5,            # đường ngăn giữa các ô
    cbar_kws={'shrink': 0.8},  # thu nhỏ thanh màu
    square=True                # ô vuông đều
)

plt.title("Ma trận tương quan giữa các biến", fontsize=16, fontweight='bold', pad=20)
plt.show()

In [ ]:
df = df.drop(['Letter_Count', 'URL_Length', 'www_Count', 'Ques_Count'], axis=1)

In [ ]:
plt.figure(figsize=(15,13))

sns.heatmap(
    df.corr(),                 # ma trận tương quan
    annot=True,                # hiển thị số
    fmt=".2f",                 # 2 chữ số thập phân
    cmap="coolwarm",           # bảng màu (có thể thử 'YlGnBu', 'viridis', 'magma', v.v.)
    linewidths=0.5,            # đường ngăn giữa các ô
    cbar_kws={'shrink': 0.8},  # thu nhỏ thanh màu
    square=True                # ô vuông đều
)

plt.title("Ma trận tương quan giữa các biến sau khi bỏ các đặc trưng có tương quan mạnh", fontsize=16, fontweight='bold', pad=20)
plt.show()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

# 5. Áp dụng mô hình

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [ ]:
x = df.drop(columns=['Label'])
y = df['Label']
x_train, x_test,y_train,y_test = train_test_split(x,y,test_size=0.3, random_state=42)

## 5.1. KNN

In [ ]:
knn = KNeighborsClassifier()
knn.fit(x_train, y_train)
y_pred_knn = knn.predict(x_test)

In [ ]:
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print("Độ chính xác của mô hình KNN: ", accuracy_knn)
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred_knn))
conf_matrix_knn = confusion_matrix(y_test, y_pred_knn)

sns.heatmap(conf_matrix_knn, annot=True, cmap='Blues', fmt='d', cbar=False)
plt.title('Ma trận nhầm lẫn của mô hình KNN')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()

## 5.2. Decision Tree

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)
y_pred_dt = dt.predict(x_test)

In [ ]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("Độ chính xác của mô hình Decision Tree: ", accuracy_dt)
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred_dt))
conf_matrix_dt = confusion_matrix(y_test, y_pred_dt)

sns.heatmap(conf_matrix_dt, annot=True, cmap='Blues', fmt='d', cbar=False)
plt.title('Ma trận nhầm lẫn của mô hình Decision Tree')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()

## 5.3. Random Forest

In [ ]:
rf = RandomForestClassifier()
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)

In [ ]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Độ chính xác của mô hình Random Forest: ", accuracy_rf)
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred_rf))
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)


sns.heatmap(conf_matrix_rf, annot=True, cmap='Blues', fmt='d', cbar=False)
plt.title('Ma trận nhầm lẫn của mô hình Random Forest')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()

## 5.4. Logistic Regression

In [ ]:
lr = LogisticRegression()
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Độ chính xác của mô hình Logistic Regression: ", accuracy_lr)
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred_lr))
conf_matrix_lr= confusion_matrix(y_test, y_pred_lr)

sns.heatmap(conf_matrix_lr, annot=True, cmap='Blues', fmt='d', cbar=False)
plt.title('Ma trận nhầm lẫn của mô hình Logistic Regression')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()

## 5.5. Mô hình XGBoost

In [ ]:
xgb = XGBClassifier(learning_rate=0.4,max_depth=7)
xgb.fit(x_train, y_train)
y_pred_xgb = xgb.predict(x_test)

In [ ]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print("Độ chính xác của mô hình XGBoost: ", accuracy_xgb)
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred_xgb))
conf_matrix_xgb= confusion_matrix(y_test, y_pred_xgb)

sns.heatmap(conf_matrix_xgb, annot=True, cmap='Blues', fmt='d', cbar=False)
plt.title('Ma trận nhầm lẫn của mô hình XGBoost')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()

## 5.6. Mô hình MLP

In [ ]:
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        alpha=1e-4,          
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42,
        verbose=False
    )
)

mlp.fit(x_train, y_train)
y_pred_mlp = mlp.predict(x_test)

In [ ]:
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
print("Độ chính xác của mô hình MLP: ", accuracy_mlp)
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred_mlp))
conf_matrix_mlp= confusion_matrix(y_test, y_pred_mlp)
sns.heatmap(conf_matrix_mlp, annot=True, cmap='Blues', fmt='d', cbar=False)

plt.title('Ma trận nhầm lẫn của mô hình MLP')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()

# 6. So sánh các mô hình

In [ ]:
# Tính precision, recall, f1 cho các mô hình

precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

precision_mlp = precision_score(y_test, y_pred_mlp)
recall_mlp= recall_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp)

In [ ]:
# Bảng so sánh các chỉ số đánh giá mô hình
performance_summary = pd.DataFrame({
    'Model': [ 'KNN', 'Decision Tree', 'Random Forest', 'Logistic Regression', ' XGBoost', 'MLP'],
    'Accuracy': [ accuracy_knn, accuracy_dt, accuracy_rf, accuracy_lr, accuracy_xgb, accuracy_mlp ],
    'Precision': [ precision_knn, precision_dt, precision_rf, precision_lr, precision_xgb, precision_mlp],
    'Recall': [recall_knn, recall_dt, recall_rf, recall_lr, recall_xgb, recall_mlp],
    'F1-score': [ f1_knn, f1_dt, f1_rf, f1_lr, f1_xgb, f1_mlp]
})

display(performance_summary)

In [ ]:
# Sơ đồ so sánh các accuracy giữa các mô hình
accuracies = [accuracy_knn, accuracy_dt, accuracy_rf, accuracy_lr, accuracy_xgb, accuracy_mlp]
models = ['KNN', 'Decision Tree', 'Random Forest', 'Logistic Regression', ' XGBoost', 'MLP']

plt.figure(figsize=(10, 6))
plt.bar(models, accuracies, color=['blue', 'orange', 'green', 'purple', 'pink', 'brown', 'yellow'])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Accuracy của các mô hình')
plt.axhline(y=max(accuracies), color='red', linestyle='--', label='Best Model')

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
y_proba_knn = knn.predict_proba(x_test)[:, 1]
y_proba_dt = dt.predict_proba(x_test)[:, 1]
y_proba_rf = rf.predict_proba(x_test)[:, 1]
y_proba_lr = lr.predict_proba(x_test)[:, 1]
y_proba_xgb = xgb.predict_proba(x_test)[:, 1]
y_proba_mlp = mlp.predict_proba(x_test)[:, 1]

In [ ]:
y_test_bin = label_binarize(y_test, classes=np.unique(y))
n_classes = y_test_bin.shape[1]

In [ ]:
fpr_knn_micro, tpr_knn_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_knn.ravel())
roc_auc_knn_micro = auc(fpr_knn_micro, tpr_knn_micro)

fpr_dt_micro, tpr_dt_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_dt.ravel())
roc_auc_dt_micro = auc(fpr_dt_micro, tpr_dt_micro)

fpr_rf_micro, tpr_rf_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_rf.ravel())
roc_auc_rf_micro = auc(fpr_rf_micro, tpr_rf_micro)

fpr_lr_micro, tpr_lr_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_lr.ravel())
roc_auc_lr_micro = auc(fpr_lr_micro, tpr_lr_micro)

fpr_xgb_micro, tpr_xgb_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_xgb.ravel())
roc_auc_xgb_micro = auc(fpr_xgb_micro, tpr_xgb_micro)

fpr_mlp_micro, tpr_mlp_micro, _ = roc_curve(y_test_bin.ravel(), y_proba_mlp.ravel())
roc_auc_mlp_micro = auc(fpr_mlp_micro, tpr_mlp_micro)

In [ ]:
plt.figure(figsize=(10, 8))

colors = ['blue', 'orange', 'green', 'purple', 'pink', 'red', 'brown', 'yellow']

plt.plot(fpr_knn_micro, tpr_knn_micro, label=f'KNN (AUC = {roc_auc_knn_micro:.3f})', color=colors[1])
plt.plot(fpr_dt_micro, tpr_dt_micro, label=f'Decision Tree (AUC = {roc_auc_dt_micro:.3f})', color=colors[2])
plt.plot(fpr_rf_micro, tpr_rf_micro, label=f'Random Forest (AUC = {roc_auc_rf_micro:.3f})', color=colors[3])
plt.plot(fpr_lr_micro, tpr_lr_micro, label=f'Logistic Regression (AUC = {roc_auc_lr_micro:.3f})', color=colors[4])
plt.plot(fpr_xgb_micro, tpr_xgb_micro, label=f'XGBoost (AUC = {roc_auc_xgb_micro:.3f})', color=colors[5])
plt.plot(fpr_mlp_micro, tpr_mlp_micro, label=f'MLP (AUC = {roc_auc_mlp_micro:.3f})', color=colors[6])

plt.plot([0, 1], [0, 1], 'k--', label='Dự đoán ngẫu nhiên')
plt.xlabel('Tỷ lệ Dương tính Giả')
plt.ylabel('Tỷ lệ Dương tính Thực')
plt.title('Đường cong ROC so sánh các mô hình')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

Lưu mô hình XGBoost về XGB-URL.plk để phục vụ quá trình dự đoán

In [ ]:
import joblib
joblib.dump(xgb, "../API/XGB.pkl")